In [2]:
!pip install -U datasets

from datasets import load_dataset

dataset = load_dataset("stanfordnlp/imdb")

print(dataset)

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [4]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text
dataset = dataset.map(lambda x: {"clean_text": clean_text(x["text"])})

print(dataset["train"][0]["clean_text"])

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

i rented i am curiousyellow from my video store because of all the controversy that surrounded it when it was first released in i also heard that at first it was seized by us customs if it ever tried to enter this country therefore being a fan of films considered controversial i really had to see this for myselfbr br the plot is centered around a young swedish drama student named lena who wants to learn everything she can about life in particular she wants to focus her attentions to making some sort of documentary on what the average swede thought about certain political issues such as the vietnam war and race issues in the united states in between asking politicians and ordinary denizens of stockholm about their opinions on politics she has sex with her drama teacher classmates and married menbr br what kills me about i am curiousyellow is that years ago this was considered pornographic really the sex and nudity scenes are few and far between even then its not shot like some cheaply m

In [5]:
from tensorflow.keras.preprocessing.text import Tokenizer

# Create tokenizer
tokenizer = Tokenizer(num_words=10000)

# Build vocabulary
tokenizer.fit_on_texts(dataset["train"]["clean_text"])


# Convert words to numbers
X_train = tokenizer.texts_to_sequences(dataset["train"]["clean_text"])
X_test = tokenizer.texts_to_sequences(dataset["test"]["clean_text"])


print(X_train[0])

[9, 1521, 9, 237, 35, 55, 391, 1136, 79, 4, 31, 1, 7084, 11, 3270, 8, 50, 8, 13, 85, 623, 7, 9, 78, 539, 11, 30, 85, 8, 13, 32, 170, 9115, 44, 8, 121, 766, 5, 2435, 10, 708, 1539, 106, 3, 331, 4, 93, 1130, 2991, 9, 62, 66, 5, 67, 10, 15, 12, 1, 113, 6, 5861, 184, 3, 182, 3767, 470, 1460, 746, 4517, 36, 466, 5, 815, 279, 54, 68, 42, 118, 7, 811, 54, 466, 5, 1113, 39, 5, 245, 46, 423, 4, 662, 20, 48, 1, 858, 199, 42, 765, 977, 1287, 135, 14, 1, 2620, 332, 2, 1513, 1287, 7, 1, 2301, 1559, 7, 195, 2128, 6795, 2, 1871, 4, 42, 64, 4643, 20, 2372, 54, 43, 390, 16, 39, 470, 1703, 7859, 2, 998, 9511, 12, 48, 1049, 69, 42, 9, 237, 6, 11, 147, 593, 10, 13, 1130, 8114, 62, 1, 390, 2, 999, 136, 23, 164, 2, 234, 195, 53, 91, 29, 21, 317, 38, 46, 6500, 90, 4518, 130, 55, 346, 159, 8, 1560, 7, 635, 390, 2, 999, 23, 3, 642, 9933, 7, 3767, 441, 53, 4873, 4477, 64, 1450, 5, 49, 165, 440, 297, 1997, 66, 390, 136, 7, 24, 3635, 12, 9, 81, 1, 869, 15, 1, 185, 11, 98, 390, 590, 7, 1, 19, 6, 590, 15, 1540, 522

In [6]:
from tensorflow.keras.preprocessing.sequence import pad_sequences


max_length = 50

X_train = pad_sequences(
    X_train,
    maxlen=max_length,
    padding="post",
    truncating="post"
)


X_test = pad_sequences(
    X_test,
    maxlen=max_length,
    padding="post",
    truncating="post"
)


print(X_train.shape)

(25000, 50)


In [7]:
from tensorflow.keras.utils import to_categorical


y_train = to_categorical(dataset["train"]["label"])

y_test = to_categorical(dataset["test"]["label"])


print(y_train[0])

[1. 0.]


In [8]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense


vocab_size = 10000


model = Sequential([

    # Convert word IDs into vectors
    Embedding(
        input_dim=vocab_size,
        output_dim=128,
        input_length=max_length
    ),


    # Learn sequence patterns
    SimpleRNN(
        64,
        return_sequences=False
    ),


    # Output classification
    Dense(
        2,
        activation="softmax"
    )
])


model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [9]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)


history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 14s 37ms/step - accuracy: 0.6331 - loss: 0.6529 - val_accuracy: 0.2634 - val_loss: 0.8544
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 20s 36ms/step - accuracy: 0.8146 - loss: 0.4179 - val_accuracy: 0.5524 - val_loss: 0.9657
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 19s 32ms/step - accuracy: 0.9431 - loss: 0.1515 - val_accuracy: 0.5582 - val_loss: 1.3104
Epoch 4/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 11s 36ms/step - accuracy: 0.9884 - loss: 0.0399 - val_accuracy: 0.5338 - val_loss: 1.6708
Epoch 5/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 11s 36ms/step - accuracy: 0.9959 - loss: 0.0167 - val_accuracy: 0.5260 - val_loss: 2.0776


In [10]:
loss, accuracy = model.evaluate(
    X_test,
    y_test
)

print("Test Accuracy:", accuracy)

782/782 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.6776 - loss: 1.3101
Test Accuracy: 0.6776400208473206
